# Supplementary1bc feature count


In [ ]:
import gc
import os
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
from tqdm import tqdm
from multiprocessing import Pool, cpu_count
from joblib import dump, load
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
UCSF_DATA_PATH = 'data/adata_cohort1.h5ad'
NYU_DATA_PATH = 'data/adata_cohort3.h5ad'
ITN_DATA_PATH = 'data/adata_cohort_2.h5ad'

TRAINED_MODEL_DIR = 'results/classification_sle_external/results_elasticnet_C1_l1_0.5'
BINNED_PEPTIDES_PATH = f'{TRAINED_MODEL_DIR}/binned_peptides_improved.csv'
OUTPUT_DIR = 'results/classification_sle_external/feature_ablation'

FEATURE_COUNTS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 30, 40, 50, 100, 200, 300, 400, 500]

MODEL_PARAMS = {
    'C': 1.0,
    'l1_ratio': 0.5,
    'n_iterations': 20,
    'n_splits': 5,
    'n_jobs': min(20, max(1, cpu_count() - 1))
}

In [ ]:
def get_feature_importance_ranking(binned_peptides_path, annotation_h5ad_path=None):
    print("Loading binned peptides file...")
    binned_df = pd.read_csv(binned_peptides_path)
    print(f"  - {len(binned_df)} total peptides loaded")

    print("\nPeptide distribution by category:")
    crosstab = pd.crosstab(
        [binned_df['freq_bin'], binned_df['mag_bin']],
        binned_df['sign']
    )
    print(crosstab)

    tier1 = binned_df[
        (binned_df['freq_bin'] == "HighFreq") &
        (binned_df['mag_bin'] == "HighMag") &
        (binned_df['sign'] == "positive")
    ].sort_values(by='median_coefficient', ascending=False).copy()
    tier1['tier'] = 1

    tier2 = binned_df[
        (binned_df['freq_bin'] == "HighFreq") &
        (binned_df['mag_bin'] == "HighMag") &
        (binned_df['sign'] == "negative")
    ].sort_values(by='median_coefficient', ascending=True).copy()
    tier2['tier'] = 2

    tier3 = binned_df[
        (binned_df['freq_bin'] == "HighFreq") &
        (binned_df['mag_bin'] == "MedMag") &
        (binned_df['sign'] == "positive")
    ].sort_values(by='median_coefficient', ascending=False).copy()
    tier3['tier'] = 3

    tier4 = binned_df[
        (binned_df['freq_bin'] == "HighFreq") &
        (binned_df['mag_bin'] == "MedMag") &
        (binned_df['sign'] == "negative")
    ].sort_values(by='median_coefficient', ascending=True).copy()
    tier4['tier'] = 4

    tier5 = binned_df[
        (binned_df['freq_bin'] == "ModerateFreq") &
        (binned_df['mag_bin'] == "HighMag")
    ].copy()
    tier5['abs_coef'] = tier5['median_coefficient'].abs()
    tier5 = tier5.sort_values(by='abs_coef', ascending=False)
    tier5['tier'] = 5

    used_peptides = set(tier1['peptide']) | set(tier2['peptide']) | set(tier3['peptide']) |\
                    set(tier4['peptide']) | set(tier5['peptide'])
    tier6 = binned_df[~binned_df['peptide'].isin(used_peptides)].copy()
    tier6['abs_coef'] = tier6['median_coefficient'].abs()
    tier6 = tier6.sort_values(by='abs_coef', ascending=False)
    tier6['tier'] = 6

    importance_df = pd.concat([tier1, tier2, tier3, tier4, tier5, tier6], ignore_index=True)
    importance_df['rank'] = range(1, len(importance_df) + 1)

    if 'abs_coef' in importance_df.columns:
        importance_df = importance_df.drop(columns=['abs_coef'])

    print(f"\nTier distribution:")
    print(f"  Tier 1 (HighFreq+HighMag+positive): {len(tier1)} peptides")
    print(f"  Tier 2 (HighFreq+HighMag+negative): {len(tier2)} peptides")
    print(f"  Tier 3 (HighFreq+MedMag+positive):  {len(tier3)} peptides")
    print(f"  Tier 4 (HighFreq+MedMag+negative):  {len(tier4)} peptides")
    print(f"  Tier 5 (ModerateFreq+HighMag):      {len(tier5)} peptides")
    print(f"  Tier 6 (remaining):                 {len(tier6)} peptides")

    print(f"\nTop 10 peptides by importance (Tier 1):")
    cols_to_show = ['rank', 'peptide', 'median_coefficient', 'selection_frequency', 'tier']
    cols_available = [c for c in cols_to_show if c in importance_df.columns]
    print(importance_df.head(10)[cols_available].to_string(index=False))

    if annotation_h5ad_path and os.path.exists(annotation_h5ad_path):
        print(f"\nAdding gene annotations from: {annotation_h5ad_path}")
        adata_var = ad.read_h5ad(annotation_h5ad_path, backed='r')
        gene_map = pd.DataFrame({'seq_id': adata_var.var_names.astype(str),
                                 'gene': adata_var.var['gene'].astype(str).values})
        adata_var.file.close()
        gene_map['fragment'] = gene_map['seq_id'].str.extract(r'fragment_(\d+)').astype(int)
        gene_map['isoform'] = gene_map['seq_id'].str.extract(r'isoform_(\d+)')

        gene_counts = gene_map.groupby('gene').size()
        single_peptide_genes = gene_counts[gene_counts == 1].index
        gene_map['isoform'] = np.where(
            gene_map['gene'].isin(single_peptide_genes),
            np.nan, 'iso' + gene_map['isoform'].fillna(''))
        gene_map['isoform'] = gene_map['isoform'].replace('iso', np.nan)

        gene_map['gene_fragment'] = gene_map['gene'] + '_' + gene_map['fragment'].astype(str)
        gene_map['pep_short'] = np.where(
            gene_map['isoform'].isna(),
            gene_map['gene_fragment'],
            gene_map['gene_fragment'] + '_' + gene_map['isoform'])

        importance_df = importance_df.merge(
            gene_map[['seq_id', 'gene', 'pep_short']],
            left_on='peptide', right_on='seq_id', how='left'
        )

        print(f"\nTop 10 peptides with gene names:")
        print(importance_df.head(10)[['rank', 'pep_short', 'gene', 'median_coefficient', 'tier']].to_string(index=False))

    return importance_df

def donor_based_cv_split_simple(sample_index, donor_ids, n_splits=5, random_state=42):
    unique_donors = np.array(donor_ids.unique())

    np.random.seed(random_state)
    np.random.shuffle(unique_donors)
    fold_size = len(unique_donors) // n_splits
    folds = []

    for i in range(n_splits):
        if i == n_splits - 1:
            val_donors = unique_donors[i*fold_size:]
        else:
            val_donors = unique_donors[i*fold_size:(i+1)*fold_size]

        train_donors = np.array([d for d in unique_donors if d not in val_donors])

        train_indices = sample_index[donor_ids.isin(train_donors)].tolist()
        val_indices = sample_index[donor_ids.isin(val_donors)].tolist()

        folds.append((train_indices, val_indices))

    return folds

def train_single_iteration_topk(args):
    seed, X, y, donor_ids, C, l1_ratio, n_splits = args

    gc.collect()

    n_pos = sum(y)
    n_neg = len(y) - n_pos
    class_ratio = n_pos / n_neg
    class_weights = {0: 1, 1: 1/class_ratio} if class_ratio < 1 else {0: class_ratio, 1: 1}

    cv_folds = donor_based_cv_split_simple(X.index, donor_ids, n_splits=n_splits, random_state=seed)

    models = []
    results = []

    for fold_idx, (train_idx, val_idx) in enumerate(cv_folds):
        X_train = X.loc[train_idx]
        X_val = X.loc[val_idx]
        train_mask = X.index.isin(train_idx)
        val_mask = X.index.isin(val_idx)
        y_train = y[train_mask]
        y_val = y[val_mask]

        model = LogisticRegression(
            penalty='elasticnet', solver='saga',
            l1_ratio=l1_ratio, C=C,
            class_weight=class_weights,
            max_iter=200, tol=1e-3,
            random_state=seed + fold_idx,
            n_jobs=1
        )

        model.fit(X_train, y_train)
        models.append(model)

        y_pred_proba = model.predict_proba(X_val)[:, 1]
        results.append({
            'seed': seed,
            'fold': fold_idx,
            'auroc': roc_auc_score(y_val, y_pred_proba),
            'n_features': X.shape[1]
        })

    return seed, models, results

def train_models_with_topk_features(adata, top_k_features, output_subdir,
                                     C=1.0, l1_ratio=0.5,
                                     n_iterations=20, n_splits=5, n_jobs=10):
    os.makedirs(output_subdir, exist_ok=True)

    k = len(top_k_features)
    print(f"\n{'='*60}")
    print(f"Training models with top {k} features")
    print(f"{'='*60}")

    print("Extracting feature subset...")
    adjusted_fc = adata.to_df(layer="adjusted_fc_over_ag")
    X = adjusted_fc[top_k_features].copy()
    X[np.isnan(X) | np.isinf(X)] = 0

    HC_samples = np.array(adata[adata.obs.group == 'healthy_control'].obs.index)
    y = ~(X.index.isin(HC_samples))
    y = np.array(y)

    donor_ids = adata.obs.loc[X.index, 'unique_patient_id']

    print(f"  - X shape: {X.shape}")
    print(f"  - Positive samples: {sum(y)}, Negative samples: {len(y) - sum(y)}")
    print(f"  - Unique donors: {donor_ids.nunique()}")

    del adjusted_fc
    gc.collect()

    seeds = [42 + i * 1000 for i in range(n_iterations)]
    args_list = [(seed, X, y, donor_ids, C, l1_ratio, n_splits) for seed in seeds]

    all_models = {}
    all_results = []

    print(f"Running {n_iterations} iterations with {n_jobs} parallel workers...")

    with Pool(n_jobs) as pool:
        for seed, models, results in tqdm(
            pool.imap(train_single_iteration_topk, args_list),
            total=n_iterations,
            desc=f"Top {k} features"
        ):
            for fold_idx, model in enumerate(models):
                all_models[f"seed_{seed}_fold_{fold_idx}"] = model
            all_results.extend(results)

    results_df = pd.DataFrame(all_results)

    dump(all_models, os.path.join(output_subdir, 'models.joblib'))
    results_df.to_csv(os.path.join(output_subdir, 'cv_results.csv'), index=False)
    with open(os.path.join(output_subdir, 'feature_names.txt'), 'w') as f:
        f.write('\n'.join(top_k_features))

    print(f"CV AUROC: {results_df['auroc'].mean():.4f} ± {results_df['auroc'].std():.4f}")

    return {
        'models': all_models,
        'results_df': results_df,
        'feature_names': top_k_features
    }

def evaluate_on_test_cohort(models, feature_names, adata_test, cohort_name='test',
                            layer='log_fold_change_over_AG',
                            label_col='group', label_value='lupus',
                            donor_col='unique_subject_id'):
    X_test = adata_test.to_df(layer=layer)

    missing_features = [f for f in feature_names if f not in X_test.columns]
    if missing_features:
        print(f"  Warning: {len(missing_features)} features not in test data, setting to 0")
        for f in missing_features:
            X_test[f] = 0

    X_test = X_test[feature_names]
    X_test[np.isnan(X_test) | np.isinf(X_test)] = 0

    y_test = (adata_test.obs[label_col] == label_value).astype(int)

    results = []
    roc_data = []
    all_predictions = []

    for model_name, model in models.items():
        pred_proba = model.predict_proba(X_test)[:, 1]
        all_predictions.append(pred_proba)

        fpr, tpr, thresholds = roc_curve(y_test, pred_proba)
        auroc = roc_auc_score(y_test, pred_proba)
        auprc = average_precision_score(y_test, pred_proba)

        seed = int(model_name.split('_')[1])
        fold = int(model_name.split('_')[3])

        results.append({
            'model': model_name,
            'seed': seed,
            'fold': fold,
            'auroc': auroc,
            'auprc': auprc
        })

        roc_data.append({
            'model': model_name,
            'fpr': fpr,
            'tpr': tpr,
            'thresholds': thresholds
        })

    results_df = pd.DataFrame(results)

    ensemble_pred_proba = np.mean(all_predictions, axis=0)
    ensemble_fpr, ensemble_tpr, _ = roc_curve(y_test, ensemble_pred_proba)

    ensemble_metrics = {
        'auroc': roc_auc_score(y_test, ensemble_pred_proba),
        'auprc': average_precision_score(y_test, ensemble_pred_proba),
        'fpr': ensemble_fpr,
        'tpr': ensemble_tpr
    }

    return {
        'results_df': results_df,
        'ensemble_metrics': ensemble_metrics,
        'roc_data': roc_data
    }

In [ ]:
def load_ucsf_training_data(ucsf_path):
    adata_ucsf = ad.read_h5ad(ucsf_path)
    nyu = ad.read_h5ad(NYU_DATA_PATH)
    nyu_control_ids = set(nyu.obs.loc[nyu.obs['group'] == 'control', 'subject_id'].astype(str))
    ucsf_hc_ids = set(adata_ucsf.obs.loc[adata_ucsf.obs['group'] == 'healthy_control', 'subject_id'].astype(str))
    shared_hc_ids = ucsf_hc_ids & nyu_control_ids
    adata_ucsf = adata_ucsf[~adata_ucsf.obs['subject_id'].astype(str).isin(shared_hc_ids)].copy()
    return adata_ucsf

def load_nyu_validation_data(nyu_path):
    return ad.read_h5ad(nyu_path)

def load_itn_validation_data(itn_path):
    adata_itn = ad.read_h5ad(itn_path)
    adata_itn.obs = adata_itn.obs.rename(columns={'sample_ID': 'patient_ID'})
    return adata_itn

def run_feature_ablation_analysis(
    ucsf_path, nyu_path, itn_path, output_dir,
    binned_peptides_path, annotation_h5ad_path=None,
    feature_counts=FEATURE_COUNTS, model_params=MODEL_PARAMS
):
    os.makedirs(output_dir, exist_ok=True)

    print("\n" + "="*70)
    print("STEP 1: Feature Importance Extraction from Binned Peptides")
    print("="*70)

    importance_df = get_feature_importance_ranking(
        binned_peptides_path,
        annotation_h5ad_path
    )
    importance_df.to_csv(os.path.join(output_dir, 'feature_importance_ranking.csv'), index=False)

    print("\n" + "="*70)
    print("STEP 2: Loading Datasets")
    print("="*70)

    print("Loading UCSF training data (minus 46 shared-HC donors)...")
    adata_ucsf = load_ucsf_training_data(ucsf_path)
    print(f"  - Shape: {adata_ucsf.shape}")
    print(f"  - HC: {(adata_ucsf.obs['group'] == 'healthy_control').sum()}")
    print(f"  - SLE: {(adata_ucsf.obs['group'] != 'healthy_control').sum()}")

    print("\nLoading NYU validation data (81 SLE / 48 HC)...")
    adata_nyu = load_nyu_validation_data(nyu_path)
    print(f"  - Shape: {adata_nyu.shape}")
    print(f"  - Donors: {adata_nyu.obs['unique_subject_id'].nunique()}")

    print("\nLoading ITN validation data (297 SLE / 91 HC)...")
    adata_itn = load_itn_validation_data(itn_path)
    print(f"  - Shape: {adata_itn.shape}")
    print(f"  - Donors: {adata_itn.obs['patient_ID'].nunique()}")

    print("\n" + "="*70)
    print("STEP 3: Training and Evaluation")
    print("="*70)

    results_by_k = {}

    for k in feature_counts:
        top_k_features = importance_df.head(k)['peptide'].tolist()
        k_label = str(k)

        tier_counts = importance_df.head(k)['tier'].value_counts().sort_index()
        print(f"\n{'#'*70}")
        print(f"Processing K = {k_label} ({k} features)")
        print(f"Tier distribution: {dict(tier_counts)}")
        print(f"{'#'*70}")

        subdir = os.path.join(output_dir, f'top_{k}_features')

        train_results = train_models_with_topk_features(
            adata_ucsf,
            top_k_features,
            subdir,
            C=model_params['C'],
            l1_ratio=model_params['l1_ratio'],
            n_iterations=model_params['n_iterations'],
            n_splits=model_params['n_splits'],
            n_jobs=model_params['n_jobs']
        )

        print(f"\nEvaluating on NYU ({adata_nyu.shape[0]} samples)...")
        nyu_results = evaluate_on_test_cohort(
            train_results['models'],
            train_results['feature_names'],
            adata_nyu,
            cohort_name='NYU',
            layer='log_fold_change_over_AG',
            label_col='group',
            label_value='lupus',
            donor_col='unique_subject_id'
        )
        print(f"  NYU AUROC: {nyu_results['results_df']['auroc'].mean():.4f} ± {nyu_results['results_df']['auroc'].std():.4f}")

        print(f"\nEvaluating on ITN ({adata_itn.shape[0]} samples)...")
        itn_results = evaluate_on_test_cohort(
            train_results['models'],
            train_results['feature_names'],
            adata_itn,
            cohort_name='ITN',
            layer='log_fold_change_over_AG',
            label_col='group',
            label_value='lupus',
            donor_col='patient_ID'
        )
        print(f"  ITN AUROC: {itn_results['results_df']['auroc'].mean():.4f} ± {itn_results['results_df']['auroc'].std():.4f}")

        nyu_results['results_df'].to_csv(os.path.join(subdir, 'nyu_results.csv'), index=False)
        itn_results['results_df'].to_csv(os.path.join(subdir, 'itn_results.csv'), index=False)
        dump(nyu_results['roc_data'], os.path.join(subdir, 'nyu_roc_data.joblib'))
        dump(itn_results['roc_data'], os.path.join(subdir, 'itn_roc_data.joblib'))

        results_by_k[k] = {
            'k_label': k_label,
            'nyu': nyu_results,
            'itn': itn_results,
            'features': top_k_features
        }

        gc.collect()

    dump(results_by_k, os.path.join(output_dir, 'all_results_by_k.joblib'))

    return results_by_k, importance_df

In [ ]:
def plot_feature_ablation_roc_combined(
    results_by_k,
    save_path=None,
    figsize=(14, 6),
    ci_level=0.95
):
    fig, axes = plt.subplots(1, 2, figsize=figsize)

    k_values = sorted(results_by_k.keys())
    n_colors = len(k_values)
    cmap = plt.cm.viridis
    colors = [cmap(i / (n_colors - 1)) for i in range(n_colors)]

    mean_fpr = np.linspace(0, 1, 1000)
    alpha_low = (1 - ci_level) / 2 * 100
    alpha_high = (1 + ci_level) / 2 * 100

    for ax, cohort, title in zip(axes, ['nyu', 'itn'], ['NYU', 'ITN']):
        for idx, k in enumerate(k_values):
            data = results_by_k[k]
            roc_data = data[cohort]['roc_data']
            results_df = data[cohort]['results_df']
            k_label = data['k_label']

            tprs = []
            for d in roc_data:
                interp_tpr = np.interp(mean_fpr, d['fpr'], d['tpr'])
                interp_tpr[0] = 0.0
                tprs.append(interp_tpr)
            tprs = np.array(tprs)

            mean_tpr = np.mean(tprs, axis=0)
            mean_tpr[-1] = 1.0

            aucs = results_df['auroc'].values
            mean_auc = np.mean(aucs)
            ci_low = np.percentile(aucs, alpha_low)
            ci_high = np.percentile(aucs, alpha_high)

            tprs_lower = np.percentile(tprs, alpha_low, axis=0)
            tprs_upper = np.percentile(tprs, alpha_high, axis=0)

            color = colors[idx]

            ax.fill_between(mean_fpr, tprs_lower, tprs_upper,
                            color=color, alpha=0.1, linewidth=0)

            label = f'Top {k_label}: {mean_auc:.2f} ({ci_low:.2f}–{ci_high:.2f})'
            ax.plot(mean_fpr, mean_tpr, color=color, linewidth=2, label=label)

        ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, alpha=0.5)
        ax.set_xlim([-0.02, 1.02])
        ax.set_ylim([-0.02, 1.02])
        ax.set_xlabel('1 − Specificity', fontsize=13)
        ax.set_ylabel('Sensitivity', fontsize=13)
        ax.set_title(f'{title} Validation', fontsize=14, fontweight='medium')
        ax.legend(loc='lower right', fontsize=9, framealpha=0.95)
        ax.grid(True, alpha=0.2)
        ax.set_aspect('equal', adjustable='box')

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"Saved: {save_path}")

    plt.show()
    return fig, axes

def plot_auc_vs_features(results_by_k, save_path=None, figsize=(10, 6)):
    k_values = sorted(results_by_k.keys())

    nyu_aucs, nyu_ci_low, nyu_ci_high = [], [], []
    itn_aucs, itn_ci_low, itn_ci_high = [], [], []

    for k in k_values:
        data = results_by_k[k]

        nyu_vals = data['nyu']['results_df']['auroc'].values
        nyu_mean = np.mean(nyu_vals)
        nyu_aucs.append(nyu_mean)
        if np.std(nyu_vals) < 1e-10:
            nyu_ci_low.append(nyu_mean); nyu_ci_high.append(nyu_mean)
        else:
            nyu_ci_low.append(np.percentile(nyu_vals, 2.5))
            nyu_ci_high.append(np.percentile(nyu_vals, 97.5))

        itn_vals = data['itn']['results_df']['auroc'].values
        itn_mean = np.mean(itn_vals)
        itn_aucs.append(itn_mean)
        if np.std(itn_vals) < 1e-10:
            itn_ci_low.append(itn_mean); itn_ci_high.append(itn_mean)
        else:
            itn_ci_low.append(np.percentile(itn_vals, 2.5))
            itn_ci_high.append(np.percentile(itn_vals, 97.5))

    nyu_aucs, nyu_ci_low, nyu_ci_high = map(np.array, (nyu_aucs, nyu_ci_low, nyu_ci_high))
    itn_aucs, itn_ci_low, itn_ci_high = map(np.array, (itn_aucs, itn_ci_low, itn_ci_high))

    nyu_yerr_low = np.maximum(nyu_aucs - nyu_ci_low, 0)
    nyu_yerr_high = np.maximum(nyu_ci_high - nyu_aucs, 0)
    itn_yerr_low = np.maximum(itn_aucs - itn_ci_low, 0)
    itn_yerr_high = np.maximum(itn_ci_high - itn_aucs, 0)

    fig, ax = plt.subplots(figsize=figsize)

    ax.errorbar(k_values, nyu_aucs, yerr=[nyu_yerr_low, nyu_yerr_high],
                marker='o', markersize=8, linewidth=2, capsize=4,
                color='#90719f', label='NYU', alpha=0.9)
    ax.errorbar(k_values, itn_aucs, yerr=[itn_yerr_low, itn_yerr_high],
                marker='s', markersize=8, linewidth=2, capsize=4,
                color='#466c4b', label='ITN', alpha=0.9)

    ax.set_xscale('log')
    ax.set_xlabel('Number of Features (K)', fontsize=14)
    ax.set_ylabel('AUROC', fontsize=14)
    ax.set_title('Diagnostic Performance vs. Feature Count', fontsize=15)
    ax.legend(fontsize=12, loc='lower right')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_ylim([0.6, 0.95])

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"Saved: {save_path}")

    plt.show()
    return fig, ax

def create_summary_table(results_by_k, save_path=None):
    rows = []

    for k in sorted(results_by_k.keys()):
        data = results_by_k[k]
        nyu_vals = data['nyu']['results_df']['auroc'].values
        itn_vals = data['itn']['results_df']['auroc'].values

        if np.std(nyu_vals) < 1e-10:
            nyu_ci = "(N/A - no variance)"
        else:
            nyu_ci = f"({np.percentile(nyu_vals, 2.5):.3f}–{np.percentile(nyu_vals, 97.5):.3f})"

        if np.std(itn_vals) < 1e-10:
            itn_ci = "(N/A - no variance)"
        else:
            itn_ci = f"({np.percentile(itn_vals, 2.5):.3f}–{np.percentile(itn_vals, 97.5):.3f})"

        rows.append({
            'K': k,
            'NYU Mean': f"{np.mean(nyu_vals):.3f}",
            'NYU SD': f"{np.std(nyu_vals):.3f}",
            'ITN Mean': f"{np.mean(itn_vals):.3f}",
            'ITN SD': f"{np.std(itn_vals):.3f}",
        })

    df = pd.DataFrame(rows)

    if save_path:
        df.to_csv(save_path, index=False)
        print(f"Saved: {save_path}")

    print("\nSummary Table:")
    print(df.to_string(index=False))

    return df

In [ ]:
print("\n" + "="*70)
print("FEATURE ABLATION ANALYSIS FOR SLE DIAGNOSTIC CLASSIFIER")
print("="*70)

results_path = os.path.join(OUTPUT_DIR, 'all_results_by_k.joblib')

if os.path.exists(results_path):
    print(f"\nLoading existing results from: {results_path}")
    results_by_k = load(results_path)
    importance_df = pd.read_csv(os.path.join(OUTPUT_DIR, 'feature_importance_ranking.csv'))
else:
    results_by_k, importance_df = run_feature_ablation_analysis(
        ucsf_path=UCSF_DATA_PATH,
        nyu_path=NYU_DATA_PATH,
        itn_path=ITN_DATA_PATH,
        output_dir=OUTPUT_DIR,
        binned_peptides_path=BINNED_PEPTIDES_PATH,
        annotation_h5ad_path=UCSF_DATA_PATH,
        feature_counts=FEATURE_COUNTS,
        model_params=MODEL_PARAMS
    )

print("\n" + "="*70)
print("GENERATING FIGURES")
print("="*70)

plot_feature_ablation_roc_combined(
    results_by_k,
    save_path=os.path.join(OUTPUT_DIR, 'supp_fig1_feature_ablation_roc_combined.pdf')
)

plot_auc_vs_features(
    results_by_k,
    save_path=os.path.join(OUTPUT_DIR, 'fig_auc_vs_features.pdf')
)

summary_df = create_summary_table(
    results_by_k,
    save_path=os.path.join(OUTPUT_DIR, 'summary_table.csv')
)

print("\n" + "="*70)
print("ANALYSIS COMPLETE")
print("="*70)
print(f"\nAll outputs saved to: {OUTPUT_DIR}")